In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
try:
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )
    print(f"Language model initialized: {llm.model}")
except Exceprion as e:
    print(f"Error initializing language model: {e}")

Language model initialized: gemini-2.5-flash


In [18]:
# Simulated Sub-Agent Handlers 

def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request"""
    print("--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."

def info_handler(request: str) -> str:
    """Simulates the Info Agent handling a request"""
    print("--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."

def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delagated"""
    print("--- HANDLING UNCLEAR REQUAST ---")
    return f"Coordinator could not delagate request: '{request}'. Please clarify."

In [24]:
# Define Coordinator Router Chain: 
# This chain decides which handler to delagate to
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
        - If the request is related to booking flights or hotels, output 'booker'.
        - For all other travel related information questions, output 'info'.
        - If the request is unclear or doesn't fit either category, output 'unclear'.
    Only output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}")
])

In [25]:
coordinator_router_prompt

ChatPromptTemplate(input_variables=['request'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Analyze the user's request and determine which specialist handler should process it.\n        - If the request is related to booking flights or hotels, output 'booker'.\n        - For all other travel related information questions, output 'info'.\n        - If the request is unclear or doesn't fit either category, output 'unclear'.\n    Only output one word: 'booker', 'info', or 'unclear'."), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['request'], input_types={}, partial_variables={}, template='{request}'), additional_kwargs={})])

In [26]:
# Define Router Chain
coordinator_router_chain = (
    coordinator_router_prompt 
    | llm 
    | StrOutputParser()
)

# Define Delegation Logic
# Use RunnableBranch to route based on the router chain's output.
branches = {
    "booker": RunnablePassthrough.assign(output = lambda x: booking_handler(x['request']['request'])),
    "info": RunnablePassthrough.assign(output = lambda x: info_handler(x['request']['request'])),
    "unclear": RunnablePassthrough.assign(output = lambda x: unclear_handler(x['request']['request'])),
}

# Create RunnableBranch: take the output of the router chain and routes the original input ('request') to the corresponding handler.
delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches['booker']),
    (lambda x: x['decision'].strip() == 'info', branches['info']),
    branches['unclear']    # Default Branch
)

# Combine the router chain and the delegation branch into a single runnable
coordinator_agent = (
    {
        "decision": coordinator_router_chain,
        "request": RunnablePassthrough()
    }
    | delegation_branch
    | (lambda x: x['output'])
)

In [27]:
def example_run(request: str) -> str:
    result = coordinator_agent.invoke({"request": request})
    print(result)

In [28]:
example_run("Book me a flight to Sydney.")

--- DELEGATING TO BOOKING HANDLER ---
Booking Handler processed request: 'Book me a flight to Sydney.'. Result: Simulated booking action.


In [29]:
example_run("What is the capital of Italy?")

--- DELEGATING TO INFO HANDLER ---
Info Handler processed request: 'What is the capital of Italy?'. Result: Simulated information retrieval.


In [31]:
example_run("Tell me about Fermi estimates.")

--- HANDLING UNCLEAR REQUAST ---
Coordinator could not delagate request: 'Tell me about Fermi estimates.'. Please clarify.
